
# U-Net Architecture Overview

Обучим собственную U-Net для задачи Image Colorization.

U-Net — это свёрточная архитектура типа **encoder-decoder**, первоначально предложенная для сегментации изображений. Она состоит из двух основных частей: encoder, который постепенно уменьшает пространственное разрешение и извлекает признаки, и decoder, который восстанавливает разрешение для получения попиксельных предсказаний.

**Почему U-Net подходит для задачи колоризации?**

Это архитектура, которая способна преобразовывать одно пространственное представление изображения в другое.

В задаче сегментации мы передаём на вход RGB-изображение и получаем на выходе предсказания для каждого пикселя. При многоклассовой сегментации выход модели имеет форму `(B, C, H, W)`, где `C` — количество классов. С помощью `softmax` можно получить вероятности классов, а `argmax` позволяет выбрать класс для каждого пикселя и сформировать двумерную маску `(H, W)`.

В задаче колоризации мы передаём чёрно-белое изображение и хотим получить на выходе два канала с предсказанными значениями `a` и `b`. Получается, что обе задачи требуют построить новое изображение, сохранив пространственное соответствие с исходным.

Допустим, у нас есть конкретный пиксель со значением `L = 50`. Можем ли мы только по этому числу определить, какого цвета должен быть пиксель? Нет! Одному значению lightness может соответствовать множество разных цветов.

Поэтому модель должна учитывать не только значение конкретного пикселя, но и контекст вокруг него: текстуры, границы, расположение объектов и общую структуру изображения.

$$
f(L) = \widehat{ab}
$$

Здесь `L` — вся карта lightness изображения, а $\widehat{ab}$ — предсказанные цветовые каналы.

**Особенность U-Net — skip connections.** Они соединяют соответствующие уровни encoder и decoder. Промежуточные feature maps из encoder передаются напрямую в decoder и объединяются с его признаками через конкатенацию по каналам. Это позволяет использовать одновременно высокоуровневый контекст и пространственные детали изображения, которые могли потеряться при уменьшении разрешения.

В нашей задаче модель будет получать нормализованный канал `L` (lightness) и предсказывать два цветовых канала `ab`. Таким образом, в отличие от многоклассовой сегментации, мы решаем **задачу регрессии**, в которой для каждого пикселя необходимо предсказать непрерывные значения цвета.

**Размерности входа и выхода:**

- Input: `(B, 1, 128, 128)` — нормализованный L-канал (lightness).
- Output: `(B, 2, 128, 128)` — предсказанные каналы ab.


# 1. Imports

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torch import nn
from image_colorization.unet import UNet
from image_colorization.dataset import ColorizationDataset

# 2. Hyperparameters & Config

In [ ]:
NUM_EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
BASE_CHANNELS = 32

config = {
    "architecture": "UNet",
    "base_channels": BASE_CHANNELS,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "AdamW",
    "loss": "L1"
}

In [ ]:
import wandb

wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: arzun9530 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!wget -c https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar -O /content/VOCtrainval_11-May-2012.tar
!tar -xf /content/VOCtrainval_11-May-2012.tar -C /content/image-colorization

In [ ]:
%cd /content/image-colorization

/content/image-colorization


# 3. Dataset & DataLoader

In [ ]:
from pathlib import Path


images_dir = Path("VOCdevkit/VOC2012/JPEGImages")
train_path = Path("data/splits/train.txt")
val_path = Path("data/splits/val.txt")

train_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128)
])
train_dataset = ColorizationDataset(images_dir=images_dir,
                                    split_path = train_path,
                                    transform = train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

val_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128)
])
val_dataset = ColorizationDataset(images_dir=images_dir,
                                  split_path = val_path,
                                  transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# 4. Training Functions

In [ ]:
from tqdm.auto import tqdm

def training_epoch(model, optimizer, criterion, train_loader, device):
    total_loss = 0.0
    total_samples = 0

    model.train()
    for L, ab in train_loader:
        L = L.to(device)
        ab = ab.to(device)

        optimizer.zero_grad()
        pred = model(L)
        loss = criterion(pred, ab)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * L.size(0)
        total_samples += L.size(0)

    return total_loss / total_samples


def validation_epoch(model, criterion, val_loader, device):
    total_loss = 0.0
    total_samples = 0
    model.eval()

    with torch.no_grad():
        for L, ab in val_loader:
            L = L.to(device)
            ab = ab.to(device)

            pred = model(L)
            loss = criterion(pred, ab)
            total_loss += loss.item() * L.size(0)
            total_samples += L.size(0)

    return total_loss / total_samples

In [ ]:
from pathlib import Path

checkpoint_dir = Path("/content/drive/MyDrive/image-colorization/checkpoints")
experiment_name = "unet_baseline_v1"
last_path = checkpoint_dir / f"{experiment_name}_last.pt"
best_path = checkpoint_dir / f"{experiment_name}_best.pt"

In [ ]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epoches, device, run=None):

    # Инициализация
    train_loss = []
    val_loss = []
    best_val_loss = float("inf")

    # Цикл обучения по эпохам
    pbar = tqdm(range(1, num_epoches + 1), desc="Training")

    for epoch in pbar:

        # Обучение и валидация
        train_loss.append(training_epoch(model, optimizer,
                                         criterion, train_loader, device))

        val_loss.append(validation_epoch(model, criterion,
                                         val_loader, device))

        # Формирование checkpoint
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_losses": train_loss,
            "val_losses": val_loss
        }

        #Сохранение последнего состояния модели
        torch.save(checkpoint, last_path)

        # Сохранение лучшей модели по validation loss
        if val_loss[-1] < best_val_loss:
            best_val_loss = val_loss[-1]
            torch.save(checkpoint, best_path)

        # Логирование метрик в W&B
        if run is not None:
            run.log({
                "epoch": epoch,
                "train_loss": train_loss[-1],
                "val_loss": val_loss[-1]
            })

        # Обновление прогресса обучения в tqdm
        pbar.set_postfix(
            train_loss=f"{train_loss[-1]:.4f}",
            val_loss=f"{val_loss[-1]:.4f}"
        )

    #Возврат истории обучения
    return train_loss, val_loss

Перед полноценным экспериментом была проверена работоспособность обучения

# 5. Model Initialization

In [ ]:
from image_colorization.unet import UNet

model = UNet().to(device)
optimizer = torch.optim.AdamW( model.parameters(),lr=LEARNING_RATE)
criterion = nn.L1Loss()

# 6. W&B Initialization

In [ ]:
run = wandb.init(
    entity="arzushkaB",
    project="image-colorization",
    name="unet-baseline",
    config=config
)

# 7. Training

In [ ]:
train_losses, val_losses = train(
    model, criterion, optimizer,
    train_loader, val_loader,
    NUM_EPOCHS, device,
    run=run
)

Training:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
run.finish()

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▇▆▆▆▅▄▄▃▃▃▃▃▂▂▂▁▁▁▁
val_loss,██▆▆▆▅▄▄▄▃▃▄▄▄▂▂▂▂▁▁
epoch,20
train_loss,0.06448
val_loss,0.06289
